In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS workspace.gold COMMENT 'Capa gold'")

In [0]:
#spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_ciuadades")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_ciudades (
  id_ciudad LONG,
  nombre_ciudad STRING,
  estado_ciudad STRING,
  region_ciudad STRING
)
""")


In [0]:
import pandas as pd

df = spark.table("workspace.silver.adidas_us_sales").toPandas()

# Se crea la dimension de ciudades, desde la tabla de ventas

# Se quita los duplicados de estas tres columnas
dim_ciudad = df[['ciudad', 'estado', 'region']].drop_duplicates()

# Se ordenan por region, estado y ciudad
dim_ciudad = dim_ciudad.sort_values(['region', 'estado', 'ciudad']).reset_index(drop=True)

# Se asigna un id incremental
dim_ciudad['id_ciudad'] = dim_ciudad.index + 1

# Se renombra las columnas
dim_ciudad = dim_ciudad.rename(columns={'ciudad': 'nombre_ciudad', 'estado': 'estado_ciudad', 'region': 'region_ciudad'})

# Se ordenan por id_ciudad
dim_ciudad = dim_ciudad[["id_ciudad", "nombre_ciudad", "estado_ciudad", "region_ciudad"]]

df_spark = spark.createDataFrame(dim_ciudad)


In [0]:

# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_ciudades")